In [6]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer
from sklearn.multioutput import MultiOutputRegressor
import numpy as np
from sklearn.metrics import root_mean_squared_error, r2_score
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('/Users/abhaypai/Library/Mobile Documents/com~apple~CloudDocs/Job stuff/Pre 2026/Projects/Portfolio Projects/RBI/Data/CFPB_Labeled_Complaints_FilledOnly.csv').drop_duplicates()

In [4]:
embedder = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
embeddings = embedder.encode(
    df["Narrative"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [7]:

# X = embeddings
X = embeddings

# Three targets together
y = df[["Severity", "Vulnerability"]]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

regressor = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    verbose=0
)

model = MultiOutputRegressor(regressor)

model.fit(X_train, y_train)

preds = model.predict(X_test)

# MAE for each target
severity_mae = mean_absolute_error(
    y_test["Severity"],
    preds[:, 0]
)

vulnerability_mae = mean_absolute_error(
    y_test["Vulnerability"],
    preds[:, 1]
)


print("Severity MAE:", severity_mae)
print("Vulnerability MAE:", vulnerability_mae)

# Overall MAE
overall_mae = np.mean([
    severity_mae,
    vulnerability_mae
])

print("Overall MAE:", overall_mae)

Severity MAE: 0.6704367960638065
Vulnerability MAE: 0.9996179162015758
Overall MAE: 0.8350273561326911


In [10]:
df[
    ["Severity",
     "Vulnerability"]
].corr(method="spearman")

,Severity,Vulnerability
Severity,1.000000,0.315626
Vulnerability,0.315626,1.000000


In [13]:
true_risk = (
    0.7 * y_test["Severity"]
    + 0.3 * y_test["Vulnerability"]
)

pred_risk = (
    0.7 * preds[:,0]
    + 0.3 * preds[:,1]
)

In [14]:
risk_mae = mean_absolute_error(
    true_risk,
    pred_risk
)

risk_r2 = r2_score(
    true_risk,
    pred_risk
)

risk_rmse = root_mean_squared_error(
    true_risk,
    pred_risk
)

print("Risk MAE:", risk_mae)
print("Risk RMSE:", risk_rmse)
print("Risk R²:", risk_r2)

Risk MAE: 0.6141699532649898
Risk RMSE: 0.8014941250613389
Risk R²: 0.6049645915833273


In [15]:
text = 'My credit report is a mixed file.'
values = model.predict(embedder.encode([text]))
values

array([[2.53150162, 1.43147949]])

In [16]:
import joblib

joblib.dump(model, "/Users/abhaypai/Library/Mobile Documents/com~apple~CloudDocs/Job stuff/Pre 2026/Projects/Portfolio Projects/RBI/Models/risk_model.joblib")

['/Users/abhaypai/Library/Mobile Documents/com~apple~CloudDocs/Job stuff/Pre 2026/Projects/Portfolio Projects/RBI/Models/risk_model.joblib']